In [1]:
"""
VELOOP Task 01 - reproducible profiling starter.
All supplied data is SIMULATED/FICTIONAL. Do not present results as production metrics.
Run: python analyze_veloop.py --behavior "VELOOP_Simulated_User_Behavior_1000.xlsx" --rewards "VELoop_Rewards_Data_01.xlsx" --out output
"""
import argparse
from pathlib import Path
import pandas as pd


In [2]:
import sys

def main():
    ap=argparse.ArgumentParser()
    ap.add_argument("--behavior",required=True)
    ap.add_argument("--rewards",required=True)
    ap.add_argument("--out",default="output")

    # Allow running directly inside notebook by providing fallback list if sys.argv doesn't have args
    if "ipykernel_launcher" in sys.argv[0] or "colab_kernel_launcher" in sys.argv[0]:
        args=ap.parse_args([
            "--behavior", "VELOOP_Simulated_User_Behavior_1000.xlsx",
            "--rewards", "VELoop_Rewards_Data_01.xlsx",
            "--out", "output"
        ])
    else:
        args=ap.parse_args()

    out=Path(args.out); out.mkdir(parents=True,exist_ok=True)
    xls=pd.ExcelFile(args.behavior)
    profiles=[]
    for sheet in xls.sheet_names:
        df=pd.read_excel(args.behavior,sheet_name=sheet)
        profiles.append({"dataset":sheet,"rows":len(df),"columns":len(df.columns),
                         "duplicate_rows":int(df.duplicated().sum()),
                         "missing_cells":int(df.isna().sum().sum()),
                         "columns_with_missing":int((df.isna().sum()>0).sum())})
    pd.DataFrame(profiles).to_csv(out/"dataset_profile.csv",index=False)
    users=pd.read_excel(args.behavior,sheet_name="01_Users_1000")
    sessions=pd.read_excel(args.behavior,sheet_name="02_Sessions")
    events=pd.read_excel(args.behavior,sheet_name="03_Feature_Events")
    tasks=pd.read_excel(args.behavior,sheet_name="05_Task_Activity")
    task_status=tasks["task_status"].astype(str).str.strip().str.lower()
    findings=pd.DataFrame([
      {"metric":"user_profiles","value":len(users),"definition":"Rows in simulated Users sheet"},
      {"metric":"sessions","value":len(sessions),"definition":"Rows in simulated Sessions sheet"},
      {"metric":"feature_events","value":len(events),"definition":"Rows in simulated Feature Events sheet"},
      {"metric":"task_activity_records","value":len(tasks),"definition":"Rows in simulated Task Activity sheet"},
      {"metric":"task_completion_rate_pct","value":round(task_status.eq("completed").mean()*100,2),"definition":"Completed task rows divided by all task rows"}
    ])
    findings.to_csv(out/"key_findings.csv",index=False)
    messy=pd.read_excel(args.rewards,sheet_name="Users_Messy_Audit")
    dq=pd.DataFrame([
      {"check":"duplicate user_id rows in messy audit","count":int(messy["user_id"].duplicated(keep=False).sum())},
      {"check":"missing cells in messy audit","count":int(messy.isna().sum().sum())}
    ])
    dq.to_csv(out/"messy_audit_checks.csv",index=False)
    print("Done. Outputs are simulated-data analyses, not production VELOOP metrics.")
if __name__=="__main__": main()

Done. Outputs are simulated-data analyses, not production VELOOP metrics.
